# YOLOv26m-cls Attention Screening

Screens YOLOv26m-cls baseline plus eight attention variants with CE + RandAugment enabled as the default YOLO classification recipe. Uses the paper dataset split, class-order audit, custom YOLO trainer, and artifact conventions.

In [ ]:
from pathlib import Path
import ast
import importlib.util
import json
import os
import shlex
import subprocess
import sys

OUTPUT_DIR = Path('/kaggle/working/shrimp_outputs_yolo_attention_screening')
LOG_DIR = Path('/kaggle/working/notebook_command_logs')
WORK_DIR = Path('/kaggle/working')
REPO_DIR = WORK_DIR / 'paper_code'
REPO_URL = 'https://github.com/hntnhan1111-ayai/CVio_Shrimp_Disease_Classification_Capstone_SU26.git'
BRANCH = 'feature/improving-lightweight-shrimp-disease-classification-coinfection-losses-randaugment'
YOLO_ATTENTION_SCREEN_EPOCHS = int(os.environ.get('YOLO_ATTENTION_SCREEN_EPOCHS', '8'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, name='command'):
    secret_values = [value for value in [globals().get('token', '')] if value]
    display_parts = []
    for part in cmd:
        text = str(part)
        for secret in secret_values:
            text = text.replace(secret, '***')
        display_parts.append(text)
    print('$', ' '.join(shlex.quote(part) for part in display_parts))
    result = subprocess.run(cmd, cwd=cwd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path = LOG_DIR / f'{name}.log'
    log_path.write_text(result.stdout, encoding='utf-8')
    print(result.stdout[-4000:])
    if result.returncode != 0:
        raise SystemExit(f'Command failed: {name}; see {log_path}')
    return result


In [ ]:
token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN') or ''
except Exception:
    token = ''

clone_url = REPO_URL
if token:
    clone_url = REPO_URL.replace('https://', f'https://x-access-token:{token}@')

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', clone_url, str(REPO_DIR)], cwd=WORK_DIR, name='git_clone')
else:
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, name='git_fetch')
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR, name='git_checkout')
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR, name='git_pull')

PROJECT_DIR = REPO_DIR / 'Improving Lightweight Shrimp Disease Classification with Co-Infection-Aware Losses and RandAugment'
if not PROJECT_DIR.exists():
    PROJECT_DIR = REPO_DIR
print('Project directory:', PROJECT_DIR)
assert (PROJECT_DIR / 'shrimp_scripts').exists(), PROJECT_DIR


In [ ]:
required = {
    'kagglehub': 'kagglehub',
    'numpy': 'numpy',
    'pandas': 'pandas',
    'PIL': 'pillow',
    'sklearn': 'scikit-learn',
    'torch': 'torch',
    'ultralytics': 'ultralytics',
    'cv2': 'opencv-python',
    'matplotlib': 'matplotlib',
    'openpyxl': 'openpyxl',
    'tqdm': 'tqdm',
}
missing = [pkg for module, pkg in required.items() if importlib.util.find_spec(module) is None]
if importlib.util.find_spec('pytorch_grad_cam') is None:
    missing.append('grad-cam')
if missing:
    run([sys.executable, '-m', 'pip', 'install', *sorted(set(missing))], cwd=PROJECT_DIR, name='pip_install_missing')
else:
    print('All required packages already importable.')


In [ ]:
NOTEBOOK_PATH = PROJECT_DIR / 'experiments/yolo_attention_screening/kaggle_yolo26m_attention_screening_t4x2.ipynb'
nb = json.loads(NOTEBOOK_PATH.read_text(encoding='utf-8'))
syntax_errors = []
for index, cell in enumerate(nb.get('cells', [])):
    if cell.get('cell_type') != 'code':
        continue
    source = ''.join(cell.get('source', []))
    try:
        ast.parse(source)
    except SyntaxError as exc:
        syntax_errors.append({'cell_index': index, 'line': exc.lineno, 'offset': exc.offset, 'message': exc.msg, 'text': (exc.text or '').strip()})
if syntax_errors:
    raise SystemExit(f'Notebook code-cell syntax errors: {syntax_errors}')

sys.path.insert(0, str(PROJECT_DIR))
from shrimp_scripts.utils import environment_versions, write_json
write_json(OUTPUT_DIR / 'environment_versions.json', environment_versions({
    'experiment': 'yolo_attention_screening',
    'epochs': YOLO_ATTENTION_SCREEN_EPOCHS,
    'repo_branch': BRANCH,
    'project_dir': str(PROJECT_DIR),
}))
print('Notebook syntax validated and environment_versions.json written.')


In [ ]:
run([sys.executable, '-m', 'compileall', 'shrimp_scripts', 'experiments/yolo_attention_screening'], cwd=PROJECT_DIR, name='compileall')
run([sys.executable, 'shrimp_scripts/run_01_prepare_dataset.py', '--output_dir', str(OUTPUT_DIR)], cwd=PROJECT_DIR, name='prepare_dataset')
run([sys.executable, 'experiments/yolo_attention_screening/run_yolo_attention_screen.py', '--output_dir', str(OUTPUT_DIR), '--epochs', str(YOLO_ATTENTION_SCREEN_EPOCHS), '--list_runs'], cwd=PROJECT_DIR, name='attention_list_runs')
run([sys.executable, 'experiments/yolo_attention_screening/run_yolo_attention_screen.py', '--output_dir', str(OUTPUT_DIR), '--epochs', str(YOLO_ATTENTION_SCREEN_EPOCHS), '--validate_resume'], cwd=PROJECT_DIR, name='attention_validate_resume')
run([sys.executable, 'experiments/yolo_attention_screening/run_yolo_attention_screen.py', '--output_dir', str(OUTPUT_DIR), '--self_test_attention_modules'], cwd=PROJECT_DIR, name='attention_module_unit_test')
run([sys.executable, 'experiments/yolo_attention_screening/run_yolo_attention_screen.py', '--output_dir', str(OUTPUT_DIR), '--epochs', str(YOLO_ATTENTION_SCREEN_EPOCHS), '--self_test_model_injection'], cwd=PROJECT_DIR, name='attention_model_injection_test')


In [ ]:
run([sys.executable, 'experiments/yolo_attention_screening/run_yolo_attention_screen.py', '--output_dir', str(OUTPUT_DIR), '--epochs', str(YOLO_ATTENTION_SCREEN_EPOCHS), '--resume'], cwd=PROJECT_DIR, name='attention_screening_train')
run([sys.executable, 'experiments/yolo_attention_screening/collect_yolo_attention_results.py', '--output_dir', str(OUTPUT_DIR), '--epochs', str(YOLO_ATTENTION_SCREEN_EPOCHS)], cwd=PROJECT_DIR, name='attention_collect_results')


In [ ]:
import pandas as pd

ranking_path = OUTPUT_DIR / 'attention_ranking.csv'
failed_path = OUTPUT_DIR / 'attention_failed_or_skipped.csv'
if not ranking_path.exists():
    raise SystemExit(f'Missing ranking artifact: {ranking_path}')
ranking = pd.read_csv(ranking_path)
if ranking.empty:
    failed_preview = failed_path.read_text(encoding='utf-8')[:4000] if failed_path.exists() else 'missing attention_failed_or_skipped.csv'
    raise SystemExit('attention_ranking.csv is empty; no valid audited completed runs were collected. Failed/skipped preview:\n' + failed_preview)
if 'none_baseline' not in set(ranking.get('attention_key', [])):
    raise SystemExit('Baseline none_baseline is absent from attention_ranking.csv; do not compare attention variants without the audited baseline.')
print(ranking.head(10).to_string(index=False))


In [ ]:
import zipfile
zip_path = Path('/kaggle/working/yolo_attention_screening_reports.zip')
include_suffixes = {'.csv', '.json', '.md'}
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file() and path.suffix.lower() in include_suffixes:
            zf.write(path, path.relative_to(OUTPUT_DIR.parent))
print('Report zip:', zip_path)
